# Training Model Multi-Class - YOLOv8s + MLflow

Notebook ini melakukan training model YOLOv8s multi-class (4 kelas) untuk proyek
**Automated Plate Count Reader** dengan tracking eksperimen menggunakan MLflow.

## Spesifikasi Training (sesuai train_18k_gpu1.py)
- **Model**: YOLOv8s (Small)
- **Kelas**: colony (0), bubble (1), dust (2), crack (3)
- **GPU**: NVIDIA RTX A4000 #1 (device=1)
- **Batch Size**: 16
- **Epochs**: 150 (early stopping patience=30)
- **Image Size**: 640
- **Workers**: 6
- **MLflow**: http://localhost:5500 / https://ml.jatnikonm.tech
- **Project**: runs/18k_multiclass_gpu1

## Setup Environment & MLflow

In [ ]:
# Install dependencies
import subprocess
import sys

deps = ['ultralytics', 'mlflow', 'pyyaml', 'matplotlib', 'opencv-python', 'numpy', 'pandas']
for dep in deps:
    try:
        __import__(dep.replace('-', '_').replace('[', '').replace(']', ''))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', dep, '-q'])

print('Dependencies ready')

In [ ]:
# Import libraries
import os
import sys
import glob
import json
import random
import subprocess
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Configuration (SESUAI train_18k_gpu1.py)
WORKSPACE = Path('/media/lambda_one/DFSSD04/project/healtcare')
YOLO_DIR = WORKSPACE / 'data' / 'yolo_18k_multiclass'  # FIXED: was yolo_multiclass
DATA_YAML = YOLO_DIR / 'data.yaml'
RUNS_DIR = WORKSPACE / 'runs'
RUN_NAME = '18k_multiclass_gpu1'  # SESUAI train_18k_gpu1.py

# Training config (SESUAI train_18k_gpu1.py)
DEVICE = '1'          # GPU 1 (RTX A4000)
BATCH = 16
EPOCHS = 150
PATIENCE = 30
IMG_SIZE = 640
WORKERS = 6
SEED = 42

# MLflow (SESUAI train_18k_gpu1.py)
MLFLOW_URI = 'http://localhost:5500'
MLFLOW_EXP_NAME = 'plate_count_reader_18k'  # FIXED: was plate_count_reader_multiclass

# Checkpoint paths
WEIGHTS_DIR = RUNS_DIR / RUN_NAME / 'weights'
BEST_MODEL = WEIGHTS_DIR / 'best.pt'
LAST_MODEL = WEIGHTS_DIR / 'last.pt'

random.seed(SEED)
np.random.seed(SEED)

print(f'Environment ready')
print(f'   Workspace : {WORKSPACE}')
print(f'   Data YAML : {DATA_YAML}')
print(f'   Output    : {RUNS_DIR / RUN_NAME}')
print(f'   Device    : GPU {DEVICE}')
print(f'   Batch     : {BATCH}')
print(f'   Epochs    : {EPOCHS}')
print(f'   MLflow    : {MLFLOW_URI}')
print(f'   Experiment: {MLFLOW_EXP_NAME}')

In [ ]:
# Setup MLflow
import mlflow
import mlflow.pytorch

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXP_NAME)

try:
    experiments = mlflow.search_experiments()
    print(f'MLflow connected: {MLFLOW_URI}')
    print(f'   Experiment: {MLFLOW_EXP_NAME}')
    print(f'   Available experiments: {[e.name for e in experiments]}')
except Exception as e:
    print(f'MLflow connection failed: {e}')
    print('   Training akan tetap berjalan tanpa MLflow tracking')

## Verifikasi Dataset

In [ ]:
# Verifikasi dataset
assert DATA_YAML.exists(), f'data.yaml tidak ditemukan: {DATA_YAML}'

import yaml
with open(str(DATA_YAML), 'r') as f:
    data_config = yaml.safe_load(f)

print('Dataset Configuration:')
print(f'   Path  : {data_config["path"]}')
print(f'   Train : {data_config["train"]}')
print(f'   Val   : {data_config["val"]}')
print(f'   Test  : {data_config["test"]}')
print(f'   NC    : {data_config["nc"]}')
print(f'   Names : {data_config["names"]}')

for split in ['train', 'val', 'test']:
    split_path = data_config['split'] if 'split' in data_config else split
    img_dir = YOLO_DIR / split / 'images'
    lbl_dir = YOLO_DIR / split / 'labels'
    n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'   {split:>5}: {n_img} images, {n_lbl} labels')

## Cek Status Training Existing

In [ ]:
# Cek apakah training sudah berjalan atau checkpoint sudah ada
import psutil

def is_training_running():
    """Cek apakah proses training YOLO sedang berjalan di GPU 1."""
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        try:
            cmdline = ' '.join(proc.info['cmdline'] or [])
            if 'train' in cmdline.lower() and ('yolo' in cmdline.lower() or 'ultralytics' in cmdline.lower()):
                return True, proc.info['pid']
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
    return False, None

training_running, training_pid = is_training_running()

print('Status Training:')
if training_running:
    print(f'   WARNING: Training sedang berjalan (PID: {training_pid})')
    print(f'   Tidak akan memulai training baru.')
elif LAST_MODEL.exists():
    print(f'   Checkpoint ditemukan: {LAST_MODEL}')
    print(f'   Akan resume training dari checkpoint terakhir.')
    print(f'   Best model: {BEST_MODEL} (exists={BEST_MODEL.exists()})')
elif BEST_MODEL.exists():
    print(f'   Best model sudah ada: {BEST_MODEL}')
    print(f'   Training sudah selesai.')
else:
    print(f'   Belum ada checkpoint. Akan mulai training dari awal.')

print(f'\nBest model exists: {BEST_MODEL.exists()}')
print(f'Last model exists: {LAST_MODEL.exists()}')

## Training dengan MLflow Tracking

In [ ]:
# Initialize model - resume jika checkpoint ada
from ultralytics import YOLO

if training_running:
    print('Training sedang berjalan di GPU 1. Tidak memulai instance baru.')
    print('Monitor via MLflow: https://ml.jatnikonm.tech')
    print('Atau gunakan cell evaluasi di bawah untuk cek model terbaru.')
elif LAST_MODEL.exists():
    print(f'Resuming dari checkpoint: {LAST_MODEL}')
    model = YOLO(str(LAST_MODEL))
    print(f'Model loaded for resume')
elif BEST_MODEL.exists():
    print(f'Training sudah selesai. Best model: {BEST_MODEL}')
    model = YOLO(str(BEST_MODEL))
    print(f'Model loaded for evaluation only')
else:
    print('Starting fresh training with YOLOv8s pretrained')
    model = YOLO('yolov8s.pt')
    print(f'Model initialized: YOLOv8s')

In [ ]:
# Training (SESUAI train_18k_gpu1.py)
# Hanya jalankan jika training tidak sedang berjalan

should_train = not training_running and not BEST_MODEL.exists()

if should_train:
    run_name = f'18k_mc_gpu1_{datetime.now().strftime("%Y%m%d_%H%M")}'
    
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f'MLflow Run ID: {run_id}')
        print(f'Run Name: {run_name}')
        print()
        
        # Log parameters
        params = {
            'model': 'yolov8s',
            'device': DEVICE,
            'batch': BATCH,
            'epochs': EPOCHS,
            'patience': PATIENCE,
            'imgsz': IMG_SIZE,
            'workers': WORKERS,
            'seed': SEED,
            'num_classes': data_config['nc'],
            'classes': str(data_config['names']),
            'resume': LAST_MODEL.exists(),
        }
        mlflow.log_params(params)
        print(f'Parameters logged: {len(params)}')
        
        # Train model (SESUAI train_18k_gpu1.py augmentation params)
        print(f'\nStarting training...')
        print(f'Device: GPU {DEVICE} | Batch: {BATCH} | Epochs: {EPOCHS}')
        print(f'Resume: {LAST_MODEL.exists()}')
        print()
        
        results = model.train(
            data=str(DATA_YAML),
            epochs=EPOCHS,
            patience=PATIENCE,
            batch=BATCH,
            imgsz=IMG_SIZE,
            device=DEVICE,
            workers=WORKERS,
            seed=SEED,
            project=str(RUNS_DIR),
            name=RUN_NAME,
            exist_ok=True,
            # Augmentation params (SESUAI train_18k_gpu1.py)
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.4,
            degrees=15,
            translate=0.1,
            scale=0.5,
            flipud=0.2,
            fliplr=0.5,
            mosaic=1.0,
            mixup=0.1,
            val=True,
            plots=True,
            save=True,
            save_period=10,
            verbose=True,
        )
        
        # Log metrics
        try:
            best_metrics = {
                'best_mAP50': float(results.results_dict.get('metrics/mAP50(B)', 0)),
                'best_mAP50-95': float(results.results_dict.get('metrics/mAP50-95(B)', 0)),
                'best_precision': float(results.results_dict.get('metrics/precision(B)', 0)),
                'best_recall': float(results.results_dict.get('metrics/recall(B)', 0)),
            }
            mlflow.log_metrics(best_metrics)
            print(f'\nBest Metrics:')
            for k, v in best_metrics.items():
                print(f'   {k}: {v}')
        except Exception as e:
            print(f'Error logging metrics: {e}')
        
        # Log model artifact
        if BEST_MODEL.exists():
            mlflow.log_artifact(str(BEST_MODEL), 'model')
            print(f'\nBest model logged: {BEST_MODEL}')
        
        # Log training plots
        run_dir = RUNS_DIR / RUN_NAME
        for fname in ['results.csv', 'confusion_matrix.png', 'results.png', 'PR_curve.png', 'F1_curve.png']:
            fpath = run_dir / fname
            if fpath.exists():
                try:
                    mlflow.log_artifact(str(fpath), 'artifacts')
                except Exception:
                    pass
        
        print(f'\nTraining selesai!')
        print(f'MLflow Run: {run_id}')
        print(f'Dashboard: https://ml.jatnikonm.tech')
else:
    if training_running:
        print('Training sedang berjalan. Skip.')
    elif BEST_MODEL.exists():
        print('Training sudah selesai. Model siap untuk evaluasi.')

## Monitor Training

Akses MLflow dashboard untuk monitoring real-time:

- **Local**: http://localhost:5500
- **Tunnel**: https://ml.jatnikonm.tech

In [ ]:
# Tampilkan training results jika ada
from IPython.display import Image as IPImage, display

plots_dir = RUNS_DIR / RUN_NAME

cm_path = plots_dir / 'confusion_matrix.png'
if cm_path.exists():
    print('Confusion Matrix:')
    display(IPImage(filename=str(cm_path)))

results_path = plots_dir / 'results.png'
if results_path.exists():
    print('Training Curves:')
    display(IPImage(filename=str(results_path)))

## Evaluasi Cepat

In [ ]:
# Quick evaluation pada test set
if BEST_MODEL.exists():
    best_model = YOLO(str(BEST_MODEL))
    
    test_results = best_model.val(
        data=str(DATA_YAML),
        split='test',
        device=DEVICE,
        imgsz=IMG_SIZE,
        verbose=True,
        plots=True,
    )
    
    print('\nHasil Evaluasi pada Test Set:')
    print(f'   mAP50    : {test_results.box.map50:.4f}')
    print(f'   mAP50-95 : {test_results.box.map:.4f}')
    print(f'   Precision : {test_results.box.mp:.4f}')
    print(f'   Recall    : {test_results.box.mr:.4f}')
    
    CLASS_NAMES = ['colony', 'bubble', 'dust', 'crack']
    print(f'\nPer Kelas:')
    for i, name in enumerate(CLASS_NAMES):
        if i < len(test_results.box.maps):
            print(f'   {name:<10}: mAP50={test_results.box.maps50[i]:.4f}, mAP50-95={test_results.box.maps[i]:.4f}')
    
    # Save best model ke lokasi final
    import shutil
    final_model_path = WORKSPACE / 'models' / 'best_multiclass_18k.pt'
    shutil.copy2(str(BEST_MODEL), str(final_model_path))
    print(f'\nModel disimpan: {final_model_path}')
    print(f'   Ukuran: {final_model_path.stat().st_size / 1024 / 1024:.2f} MB')
else:
    print(f'Best model belum tersedia. Training mungkin masih berjalan.')

## Ringkasan

In [ ]:
# Ringkasan
print('=' * 70)
print('RINGKASAN TRAINING MODEL MULTI-CLASS')
print('=' * 70)
print()
print(f'{"Parameter":<35} {"Nilai"}')
print('-' * 70)
print(f'{"Model":<35} YOLOv8s Multi-Class')
print(f'{"Dataset":<35} yolo_18k_multiclass')
print(f'{"Device":<35} GPU {DEVICE} (RTX A4000)')
print(f'{"Batch size":<35} {BATCH}')
print(f'{"Image size":<35} {IMG_SIZE}')
print(f'{"Epochs":<35} {EPOCHS}')
print(f'{"Patience":<35} {PATIENCE}')
print(f'{"Workers":<35} {WORKERS}')
print(f'{"MLflow Experiment":<35} {MLFLOW_EXP_NAME}')
print(f'{"Run dir":<35} {RUNS_DIR / RUN_NAME}')
print(f'{"Best model":<35} {BEST_MODEL} (exists={BEST_MODEL.exists()})')
print(f'{"Last model":<35} {LAST_MODEL} (exists={LAST_MODEL.exists()})')
print('=' * 70)